# ComfyUI Kaggle — Auto GPU V4.4
### T4×2 preferencial · fallback P100 · Manager v4 · HF gated · downloads server-side · Cloudflare resiliente

Esta versão parte da V4.2 e corrige os pontos que podiam produzir falsos positivos ou deixar processos órfãos.

## Principais correções da V4.4

- o patch de Hugging Face gated agora é **realmente injetado no processo do ComfyUI** via `PYTHONPATH`;
- o Launcher valida no log que o `sitecustomize.py` foi carregado pelo processo real;
- Supervisor e ComfyUI compartilham um **process group controlado**;
- existe `comfyui.pid` separado do `supervisor.pid`;
- restart mata somente o filho ComfyUI; stop/relaunch mata o grupo inteiro;
- limpeza de processos órfãos que ainda estejam usando `main.py --port 8188`;
- health-check não aceita mais um ComfyUI antigo apenas porque a porta 8188 ficou aberta;
- CORS para o Quick Tunnel fica **explicitamente habilitado**, evitando o 403 do middleware `origin_only` ao abrir a URL a partir de outro site;
- `allow_git_url_install` e `allow_pip_install` ficam desabilitados por padrão;
- o bridge respeita `ENABLE_SERVER_SIDE_DOWNLOAD_BRIDGE` de verdade;
- fallback de instalação de modelos usa a rota V2 dedicada `queue/install_model` antes de `batch`;
- `vae_approx` foi adicionado ao roteamento de modelos;
- download HF autenticado suporta resume, verificação MD5 e checagem de espaço livre quando o tamanho é conhecido;
- download do `cloudflared` é atômico;
- diagnóstico mostra PIDs, patch HF, commit do ComfyUI, Manager e storage.

## Ordem recomendada

1. **1 — CONFIGURAÇÃO**
2. **1B — HUGGING FACE AUTH**
3. **1C — VALIDAR TOKEN/GATED**
4. **2 — GPU + TORCH**
5. **3 — STORAGE**
6. **4 — INSTALAR/ATUALIZAR COMFYUI**
7. **4B — PATCH HF GATED**
8. **5 — STORAGE ROUTER**
9. **6 — MANAGER**
10. **6B — SERVER-SIDE DOWNLOAD BRIDGE**
11. **7 — CLOUDFLARED**
12. **7B — TESTE DO DOWNLOADER MANAGER**
13. **8 — LAUNCHER ROBUSTO**
14. **8B — VALIDAÇÃO**
15. **10 — KEEP-ALIVE**

> O Quick Tunnel continua sendo uma URL pública e temporária. A V4.4 reduz a superfície de risco, mas não transforma Quick Tunnel em uma camada de autenticação.


In [ ]:
# ============================================================
# 1) CONFIGURAÇÃO — V4.4
# NÃO importe torch manualmente antes da célula 2.
# ============================================================

from pathlib import Path

NOTEBOOK_VERSION = "4.3"

COMFY_DIR = Path("/kaggle/working/ComfyUI")
WORKING_ROOT = Path("/kaggle/working")
SCRATCH_ROOT = Path("/kaggle/temp/comfyui")
PIP_CACHE = Path("/kaggle/temp/pip-cache")

MODELS_DIR = SCRATCH_ROOT / "models"
INPUT_DIR = SCRATCH_ROOT / "input"
OUTPUT_DIR = SCRATCH_ROOT / "output"
TEMP_DIR = SCRATCH_ROOT / "temp"

USER_DIR = COMFY_DIR / "user"
CUSTOM_NODES_DIR = COMFY_DIR / "custom_nodes"

PORT = 8188
LISTEN_HOST = "127.0.0.1"

# ComfyUI
UPDATE_COMFYUI = True
# Opcional: tag/branch/commit. Vazio = acompanhar a branch padrão.
COMFYUI_GIT_REF = ""
ENABLE_MANAGER = True
USE_LEGACY_MANAGER_UI = False
ENABLE_ASSETS = True

# Ao abrir a URL a partir do Kaggle/outro site, o navegador pode enviar
# Sec-Fetch-Site: cross-site. O middleware origin_only atual pode responder 403.
# Por isso Quick Tunnel usa o middleware CORS explicitamente.
ENABLE_REMOTE_CORS = True
CORS_ORIGIN = "*"
ALLOW_WILDCARD_CORS = True

# Manager
MANAGER_NETWORK_MODE = "personal_cloud"
MANAGER_SECURITY_LEVEL = "normal"
# Superfícies arbitrárias de instalação: OFF por padrão.
ALLOW_GIT_URL_INSTALL = False
ALLOW_PIP_INSTALL = False

# Bridge de downloads de modelos no servidor
ENABLE_SERVER_SIDE_DOWNLOAD_BRIDGE = True

# GPU / Torch
AUTO_FIX_P100_TORCH = True
P100_TORCH_VERSION = "2.10.0"
P100_TORCHVISION_VERSION = "0.25.0"
P100_TORCHAUDIO_VERSION = "2.10.0"
P100_TORCH_INDEX = "https://download.pytorch.org/whl/cu126"

# Hugging Face
REQUIRE_HF_TOKEN = True
LOAD_HF_TOKEN_FROM_KAGGLE_SECRETS = True
HF_TOKEN_SECRET_NAME = "HF_TOKEN"
# Inline fica bloqueado por padrão para evitar salvar segredo no notebook.
ALLOW_INLINE_HF_TOKEN = False
HF_TOKEN_EXPLICIT = ""
PATCH_MANAGER_HF_GATED_DOWNLOADS = True
HF_GATED_TEST_ENABLED = True
HF_GATED_TEST_REPO = "Lightricks/LTX-2.5"
HF_GATED_TEST_FILE = ".gitattributes"
DOWNLOAD_MIN_FREE_GIB = 2.0

# Cloudflare Quick Tunnel
MAX_TUNNEL_ATTEMPTS = 4
TUNNEL_START_TIMEOUT = 90
PUBLIC_PROBE_TIMEOUT = 20
PUBLIC_PROBE_RETRIES = 6
PUBLIC_PROBE_DELAY = 3
# Vazio = release latest. Ex.: "2026.8.2" para fixar versão.
CLOUDFLARED_VERSION = ""
REFRESH_CLOUDFLARED = False

# Keep-alive
HEARTBEAT_SECONDS = 60
PUBLIC_CHECK_EVERY = 2
PUBLIC_FAILURES_BEFORE_ROTATE = 3

# Não force PYTORCH_CUDA_ALLOC_CONF. O ComfyUI escolhe o allocator antes do import de torch.
CUDA_MALLOC_MODE = "auto"  # auto mantém o comportamento padrão do ComfyUI.

EXTRA_COMFY_ARGS = []

CLOUDFLARED_BIN = WORKING_ROOT / "bin" / "cloudflared"
CLOUDFLARED_LOG = WORKING_ROOT / "cloudflared.log"
COMFY_LOG = WORKING_ROOT / "comfyui.log"
SUPERVISOR_LOG = WORKING_ROOT / "comfyui-supervisor.log"

RUNTIME_DIR = WORKING_ROOT / ".comfyui-kaggle"
SUPERVISOR_PID = RUNTIME_DIR / "supervisor.pid"
COMFY_PID = RUNTIME_DIR / "comfyui.pid"
CLOUDFLARE_PID = RUNTIME_DIR / "cloudflared.pid"
PUBLIC_URL_FILE = RUNTIME_DIR / "public_url.txt"
ACCESS_URL_FILE = RUNTIME_DIR / "access_url.txt"
STOP_FLAG = RUNTIME_DIR / "stop"
LAUNCH_ID_FILE = RUNTIME_DIR / "launch_id.txt"

for p in [
    WORKING_ROOT, SCRATCH_ROOT, PIP_CACHE, MODELS_DIR,
    INPUT_DIR, OUTPUT_DIR, TEMP_DIR, RUNTIME_DIR
]:
    p.mkdir(parents=True, exist_ok=True)

if ENABLE_REMOTE_CORS:
    if not CORS_ORIGIN:
        raise ValueError("ENABLE_REMOTE_CORS=True exige CORS_ORIGIN explícito.")
    if CORS_ORIGIN == "*" and not ALLOW_WILDCARD_CORS:
        raise ValueError(
            "CORS_ORIGIN='*' foi bloqueado na V4.4. "
            "Defina um origin específico ou ALLOW_WILDCARD_CORS=True conscientemente."
        )

print("Configuração V4.4 carregada.")
print("ComfyUI       :", COMFY_DIR)
print("Models        :", MODELS_DIR)
print("Nodes         :", CUSTOM_NODES_DIR)
print("Port          :", PORT)
print("CORS          :", CORS_ORIGIN if ENABLE_REMOTE_CORS else "OFF")
if ENABLE_REMOTE_CORS and CORS_ORIGIN == "*":
    print("⚠️ CORS=* é usado para compatibilidade com Quick Tunnel; mantenha instalações arbitrárias desligadas.")
print("Manager       :", "ON" if ENABLE_MANAGER else "OFF")
print("Git URL install:", "ON" if ALLOW_GIT_URL_INSTALL else "OFF")
print("Pip install   :", "ON" if ALLOW_PIP_INSTALL else "OFF")


In [ ]:
# ============================================================
# 1B) HUGGING FACE AUTH — V4.4
# ============================================================
# PRIORIDADE:
# 1. variável de ambiente HF_TOKEN
# 2. Kaggle Secret HF_TOKEN
# 3. token inline SOMENTE se ALLOW_INLINE_HF_TOKEN=True
# O valor do token nunca é impresso.
# ============================================================

import os
import sys
import subprocess

token = (os.environ.get("HF_TOKEN") or "").strip()
token_source = "environment" if token else None

if not token and LOAD_HF_TOKEN_FROM_KAGGLE_SECRETS:
    try:
        from kaggle_secrets import UserSecretsClient
        secret_value = UserSecretsClient().get_secret(HF_TOKEN_SECRET_NAME)
        if secret_value:
            token = secret_value.strip()
            token_source = f"Kaggle Secret:{HF_TOKEN_SECRET_NAME}"
    except Exception as e:
        print(
            "ℹ️ Kaggle Secret HF_TOKEN não encontrado/disponível:",
            type(e).__name__,
        )

if not token and ALLOW_INLINE_HF_TOKEN:
    token = (HF_TOKEN_EXPLICIT or "").strip()
    if token:
        token_source = "inline"
        print("⚠️ Token inline ativo. Evite salvar/publicar esta cópia do notebook.")

if not token:
    HF_TOKEN_AVAILABLE = False
    print("=" * 72)
    print("⚠️ HUGGING FACE TOKEN NÃO CONFIGURADO")
    print("=" * 72)
    print(f"Crie um Kaggle Secret com o nome exato: {HF_TOKEN_SECRET_NAME}")
    print("Downloads públicos continuam possíveis; gated/privados não.")
    if REQUIRE_HF_TOKEN:
        raise RuntimeError(
            "REQUIRE_HF_TOKEN=True e nenhum token foi encontrado. "
            "Configure o Kaggle Secret HF_TOKEN e execute novamente."
        )
else:
    if not token.startswith("hf_"):
        raise ValueError("O Hugging Face token encontrado não começa com 'hf_'.")

    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    os.environ["HUGGINGFACE_TOKEN"] = token

    try:
        import huggingface_hub
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"],
            check=True,
        )

    from huggingface_hub import login, whoami

    login(token=token, add_to_git_credential=False)

    try:
        info = whoami(token=token)
        HF_TOKEN_AVAILABLE = True
        username = (
            info.get("name")
            or info.get("fullname")
            or info.get("user")
            or "conta autenticada"
        )
        print("=" * 72)
        print("✅ HUGGING FACE AUTENTICADO")
        print("=" * 72)
        print("Conta        :", username)
        print("Origem token :", token_source)
        print("Token value  : [OCULTO]")
        print("O token será herdado pelos subprocessos do ComfyUI.")
    except Exception as e:
        HF_TOKEN_AVAILABLE = False
        raise RuntimeError(
            "O token foi encontrado, mas o Hugging Face recusou a autenticação. "
            f"{type(e).__name__}: {e}"
        ) from e


In [ ]:
# ============================================================
# 1C) VALIDAR TOKEN + ACESSO REAL AO REPO GATED
# ============================================================

import os
from pathlib import Path

print("=" * 78)
print("HUGGING FACE — AUTH + GATED ACCESS")
print("=" * 78)

if not HF_GATED_TEST_ENABLED:
    print("Teste gated desabilitado por configuração.")
else:
    token = (os.environ.get("HF_TOKEN") or "").strip()
    if not token:
        raise RuntimeError("HF_TOKEN não está disponível. Execute primeiro a célula 1B.")

    from huggingface_hub import whoami, hf_hub_download
    from huggingface_hub.errors import GatedRepoError, HfHubHTTPError

    account = whoami(token=token)
    print("Conta HF     :", account.get("name") or account.get("fullname") or "autenticada")
    print("Token        : ✅ válido")
    print("Repo testado :", HF_GATED_TEST_REPO)
    print("Arquivo      :", HF_GATED_TEST_FILE)
    print()

    test_dir = Path("/kaggle/temp/hf-gated-auth-test")
    test_dir.mkdir(parents=True, exist_ok=True)

    try:
        downloaded = hf_hub_download(
            repo_id=HF_GATED_TEST_REPO,
            filename=HF_GATED_TEST_FILE,
            token=token,
            local_dir=str(test_dir),
            force_download=True,
        )
        size = Path(downloaded).stat().st_size
        print("Gated access : ✅ AUTORIZADO")
        print("Arquivo local:", downloaded)
        print("Bytes        :", size)
    except GatedRepoError as e:
        raise RuntimeError(
            f"Token válido, mas a conta não tem acesso aceito a {HF_GATED_TEST_REPO}. "
            "Abra a página do modelo no Hugging Face, aceite os termos e tente novamente."
        ) from e
    except HfHubHTTPError as e:
        raise RuntimeError(
            f"Falha HTTP validando acesso gated em {HF_GATED_TEST_REPO}: {e}"
        ) from e


In [ ]:
# ============================================================
# 2) GPU + TORCH — T4x2 / P100 fallback
# ============================================================

import json
import subprocess
import sys

def sh(cmd, check=True):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), text=True, check=check)

gpu_query = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=index,name,memory.total,compute_cap",
        "--format=csv,noheader,nounits",
    ],
    text=True,
    capture_output=True,
    check=True,
)

GPU_ROWS = []
for raw in gpu_query.stdout.strip().splitlines():
    parts = [x.strip() for x in raw.split(",")]
    if len(parts) >= 4:
        GPU_ROWS.append({
            "index": int(parts[0]),
            "name": parts[1],
            "memory_mib": int(parts[2]),
            "compute_cap": parts[3],
        })

if not GPU_ROWS:
    raise RuntimeError("Nenhuma GPU NVIDIA detectada.")

print("=" * 78)
print("HARDWARE")
print("=" * 78)
for g in GPU_ROWS:
    print(
        f"GPU {g['index']}: {g['name']} | "
        f"{g['memory_mib']/1024:.2f} GiB | SM {g['compute_cap']}"
    )

CUDA_DEVICE_IDS = ",".join(str(g["index"]) for g in GPU_ROWS)
DEFAULT_GPU = GPU_ROWS[0]["index"]

HAS_P100 = any(
    "P100" in g["name"].upper() or g["compute_cap"].startswith("6.0")
    for g in GPU_ROWS
)
HAS_DUAL_T4 = (
    len(GPU_ROWS) >= 2
    and all("T4" in g["name"].upper() for g in GPU_ROWS[:2])
)

if HAS_DUAL_T4:
    print("\n✅ T4 x2 detectada.")
elif HAS_P100:
    print("\n⚠️ P100 detectada; verificando suporte sm_60.")
else:
    print(f"\nℹ️ {len(GPU_ROWS)} GPU(s) detectada(s).")

probe_code = '''
import json
try:
    import torch
    print(json.dumps({
        "ok": True,
        "version": torch.__version__,
        "cuda": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "arch_list": torch.cuda.get_arch_list() if torch.cuda.is_available() else [],
    }))
except Exception as e:
    print(json.dumps({"ok": False, "error": repr(e)}))
'''

def probe_torch():
    p = subprocess.run(
        [sys.executable, "-c", probe_code],
        text=True,
        capture_output=True,
    )
    lines = p.stdout.strip().splitlines()
    if not lines:
        return {"ok": False, "error": (p.stdout + p.stderr)[-5000:]}
    try:
        return json.loads(lines[-1])
    except Exception:
        return {"ok": False, "error": (p.stdout + p.stderr)[-5000:]}

probe = probe_torch()
print("\nTorch atual:", probe)

needs_fix = HAS_P100 and (
    not probe.get("ok")
    or "sm_60" not in probe.get("arch_list", [])
)

if needs_fix:
    if not AUTO_FIX_P100_TORCH:
        raise RuntimeError("P100 sem sm_60 na build Torch atual.")

    print("\n" + "=" * 78)
    print("INSTALANDO TORCH CUDA 12.6 PARA P100")
    print("=" * 78)

    sh([
        sys.executable, "-m", "pip", "install",
        "--disable-pip-version-check",
        "--no-input",
        "--upgrade",
        "--force-reinstall",
        "--cache-dir", str(PIP_CACHE),
        f"torch=={P100_TORCH_VERSION}",
        f"torchvision=={P100_TORCHVISION_VERSION}",
        f"torchaudio=={P100_TORCHAUDIO_VERSION}",
        "--index-url", P100_TORCH_INDEX,
    ])

    probe = probe_torch()
    print("Torch corrigido:", probe)

    if not probe.get("ok") or "sm_60" not in probe.get("arch_list", []):
        raise RuntimeError(f"sm_60 ainda ausente: {probe}")

cuda_test_code = '''
import torch, json
result = {
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "arch_list": torch.cuda.get_arch_list(),
    "devices": [],
}
for i in range(torch.cuda.device_count()):
    x = torch.ones((64, 64), device=f"cuda:{i}", dtype=torch.float16)
    y = x @ x
    torch.cuda.synchronize(i)
    result["devices"].append({
        "index": i,
        "name": torch.cuda.get_device_name(i),
        "sum": float(y.sum().item()),
    })
print(json.dumps(result))
'''

test = subprocess.run(
    [sys.executable, "-c", cuda_test_code],
    text=True,
    capture_output=True,
)

if test.returncode != 0:
    print(test.stdout)
    print(test.stderr)
    raise RuntimeError("Teste CUDA real falhou.")

CUDA_VALIDATION = json.loads(test.stdout.strip().splitlines()[-1])
print("\n✅ CUDA validado:")
print(json.dumps(CUDA_VALIDATION, indent=2))
print("\nCUDA IDs para ComfyUI:", CUDA_DEVICE_IDS)


In [ ]:
# ============================================================
# 3) STORAGE
# ============================================================

import shutil
import subprocess
from pathlib import Path

def gib(n):
    return n / (1024 ** 3)

print("=" * 78)
print("STORAGE")
print("=" * 78)

for p in [Path("/kaggle/working"), Path("/kaggle/temp"), Path("/tmp")]:
    if p.exists():
        u = shutil.disk_usage(p)
        print(
            f"{str(p):18} total={gib(u.total):7.1f} GiB | "
            f"livre={gib(u.free):7.1f} GiB"
        )

subprocess.run(
    ["df", "-h", "/kaggle/working", "/kaggle/temp", "/tmp"],
    check=False,
)


In [ ]:
# ============================================================
# 4) INSTALAR / ATUALIZAR COMFYUI — V4.4
# Protege Torch/TorchVision/TorchAudio.
# ============================================================

import re
import subprocess
import sys


def run(cmd, cwd=None, check=True, capture_output=False):
    print("\n$", " ".join(map(str, cmd)))
    return subprocess.run(
        list(map(str, cmd)),
        cwd=str(cwd) if cwd else None,
        check=check,
        text=capture_output,
        capture_output=capture_output,
    )

if not (COMFY_DIR / ".git").exists():
    run([
        "git", "clone", "--depth", "1",
        "https://github.com/Comfy-Org/ComfyUI.git",
        COMFY_DIR,
    ])

if COMFYUI_GIT_REF:
    # Funciona para branch/tag/commit alcançável pelo remoto.
    run(["git", "fetch", "--depth", "1", "origin", COMFYUI_GIT_REF], cwd=COMFY_DIR)
    run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=COMFY_DIR)
elif UPDATE_COMFYUI:
    dirty = subprocess.run(
        ["git", "status", "--porcelain"],
        cwd=COMFY_DIR,
        text=True,
        capture_output=True,
        check=True,
    ).stdout.strip()
    if dirty:
        print("⚠️ Repo possui mudanças locais; update ignorado.")
    else:
        run(["git", "pull", "--ff-only"], cwd=COMFY_DIR)

req_src = COMFY_DIR / "requirements.txt"
req_safe = COMFY_DIR / ".kaggle_requirements_safe.txt"
protected = {"torch", "torchvision", "torchaudio"}

if not req_src.exists():
    raise RuntimeError(f"requirements.txt não encontrado: {req_src}")

safe = []
for raw in req_src.read_text(encoding="utf-8").splitlines():
    line = raw.strip()
    if not line or line.startswith("#"):
        safe.append(raw)
        continue
    name = re.split(r"[<>=!~\[\s;]", line, maxsplit=1)[0].strip().lower()
    if name in protected:
        print("Protegendo stack CUDA:", line)
    else:
        safe.append(raw)

req_safe.write_text("\n".join(safe) + "\n", encoding="utf-8")

run([
    sys.executable, "-m", "pip", "install",
    "--disable-pip-version-check",
    "--cache-dir", str(PIP_CACHE),
    "-r", req_safe,
])

manager_req = COMFY_DIR / "manager_requirements.txt"
if ENABLE_MANAGER:
    if not manager_req.exists():
        raise RuntimeError("Manager habilitado, mas manager_requirements.txt não existe.")
    run([
        sys.executable, "-m", "pip", "install",
        "--disable-pip-version-check",
        "--cache-dir", str(PIP_CACHE),
        "-r", manager_req,
    ])

commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=COMFY_DIR,
    text=True,
    capture_output=True,
    check=True,
).stdout.strip()

describe = subprocess.run(
    ["git", "describe", "--tags", "--always", "--dirty"],
    cwd=COMFY_DIR,
    text=True,
    capture_output=True,
    check=False,
).stdout.strip()

print("\n✅ ComfyUI preparado.")
print("Commit :", commit)
print("Describe:", describe or "n/a")


In [ ]:
# ============================================================
# 4B) V4.4 — PATCH DO DOWNLOADER DO MANAGER PARA HF GATED
# ============================================================

import os
from pathlib import Path

SITECUSTOMIZE_DIR = WORKING_ROOT / "comfyui-kaggle-python"
SITECUSTOMIZE_DIR.mkdir(parents=True, exist_ok=True)
SITECUSTOMIZE_FILE = SITECUSTOMIZE_DIR / "sitecustomize.py"
HF_PATCH_SIGNATURE = "[Kaggle-HF-Gated-Patch] READY V4.4"

SITE_SOURCE = r'''import builtins
import hashlib
import os
import shutil
import sys
import time
from pathlib import Path
from urllib.parse import urlparse

_PREFIX = "[Kaggle-HF-Gated-Patch]"
_READY = "[Kaggle-HF-Gated-Patch] READY V4.4"

# CRITICAL STARTUP RULE:
# Do NOT import torch or torchvision from sitecustomize.
# ComfyUI must choose its CUDA allocator before torch is imported.
# Importing torchvision here imports torch too early and can produce:
# "Allocator backend parsed at runtime != allocator backend parsed at load time"
# when ComfyUI subsequently enables cudaMallocAsync.
_original_import = builtins.__import__
_tv_patched = False
_tv_original_download_url = None


def _is_hf_url(url):
    try:
        host = (urlparse(str(url)).hostname or "").lower()
        return host == "huggingface.co" or host.endswith(".huggingface.co")
    except Exception:
        return False


def _filename(url, filename):
    if filename:
        return filename
    try:
        name = Path(urlparse(str(url)).path).name
        if name:
            return name
    except Exception:
        pass
    return "download.bin"


def _token():
    return (
        os.environ.get("HF_TOKEN")
        or os.environ.get("HUGGING_FACE_HUB_TOKEN")
        or os.environ.get("HUGGINGFACE_TOKEN")
        or ""
    ).strip()


def _request(url, headers):
    import requests
    return requests.get(
        url,
        headers=headers,
        stream=True,
        allow_redirects=True,
        timeout=(30, 300),
    )


def _integrity(path, md5):
    if not md5:
        return True
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest() == md5


def _authenticated_download_url(url, root, filename=None, md5=None, max_redirect_hops=3, *args, **kwargs):
    token = _token()
    original = _tv_original_download_url

    if not _is_hf_url(url) or not token:
        return original(
            url,
            root,
            filename=filename,
            md5=md5,
            max_redirect_hops=max_redirect_hops,
        )

    root = os.path.expanduser(str(root))
    os.makedirs(root, exist_ok=True)
    name = _filename(url, filename)
    destination = os.path.join(root, name)
    partial = destination + ".part"

    if os.path.isfile(destination):
        if _integrity(destination, md5):
            print(f"{_PREFIX} Já existe e passou integridade: {destination}", flush=True)
            return
        try:
            os.remove(destination)
        except OSError:
            pass

    headers = {
        "Authorization": f"Bearer {token}",
        "User-Agent": "ComfyUI-Kaggle-HF-Gated/4.4",
        "Accept": "*/*",
    }

    resume_from = 0
    if os.path.isfile(partial):
        try:
            resume_from = os.path.getsize(partial)
        except Exception:
            resume_from = 0

    if resume_from > 0:
        headers["Range"] = f"bytes={resume_from}-"

    print(f"{_PREFIX} Download autenticado: {name} -> {root}", flush=True)
    response = _request(url, headers)

    if response.status_code == 416 and resume_from > 0:
        response.close()
        try:
            os.remove(partial)
        except OSError:
            pass
        resume_from = 0
        headers.pop("Range", None)
        response = _request(url, headers)

    if resume_from > 0 and response.status_code == 200:
        resume_from = 0

    if response.status_code not in (200, 206):
        status = response.status_code
        try:
            detail = response.text[:300]
        except Exception:
            detail = ""
        response.close()
        raise RuntimeError(
            f"{_PREFIX} Hugging Face HTTP {status} para {name}. "
            f"O token foi enviado, mas o acesso/download foi recusado. {detail}"
        )

    length = response.headers.get("content-length")
    remaining = int(length) if length and length.isdigit() else None
    total = resume_from + remaining if remaining is not None else None

    if remaining is not None:
        try:
            margin_gib = float(os.environ.get("KAGGLE_DOWNLOAD_MIN_FREE_GIB", "2"))
        except Exception:
            margin_gib = 2.0
        free = shutil.disk_usage(root).free
        required = remaining + int(margin_gib * 1024**3)
        if free < required:
            response.close()
            raise RuntimeError(
                f"{_PREFIX} Espaço insuficiente para {name}: "
                f"livre={free/1024**3:.2f} GiB, "
                f"necessário≈{required/1024**3:.2f} GiB incluindo margem."
            )

    mode = "ab" if response.status_code == 206 and resume_from > 0 else "wb"
    downloaded = resume_from
    last_report = time.time()
    try:
        with open(partial, mode) as f:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if not chunk:
                    continue
                f.write(chunk)
                downloaded += len(chunk)
                now = time.time()
                if now - last_report >= 10:
                    if total:
                        pct = downloaded * 100.0 / total
                        print(
                            f"{_PREFIX} {name}: "
                            f"{downloaded/1024**3:.2f}/{total/1024**3:.2f} GiB "
                            f"({pct:.1f}%)",
                            flush=True,
                        )
                    else:
                        print(f"{_PREFIX} {name}: {downloaded/1024**3:.2f} GiB", flush=True)
                    last_report = now
    finally:
        response.close()

    if not _integrity(partial, md5):
        try:
            os.remove(partial)
        except OSError:
            pass
        raise RuntimeError(f"{_PREFIX} Falha de integridade MD5 em {name}.")

    os.replace(partial, destination)
    print(f"{_PREFIX} Concluído: {destination}", flush=True)


def _patch_torchvision_if_available():
    global _tv_patched, _tv_original_download_url
    if _tv_patched:
        return
    module = sys.modules.get("torchvision.datasets.utils")
    if module is None or not hasattr(module, "download_url"):
        return
    _tv_original_download_url = module.download_url
    module.download_url = _authenticated_download_url
    _tv_patched = True
    launch_id = (os.environ.get("KAGGLE_LAUNCH_ID") or "none").strip()
    print(f"{_PREFIX} torchvision hook aplicado launch={launch_id}", flush=True)


def _import_hook(name, globals=None, locals=None, fromlist=(), level=0):
    module = _original_import(name, globals, locals, fromlist, level)
    if name.startswith("torchvision") or "torchvision" in sys.modules:
        _patch_torchvision_if_available()
    return module


builtins.__import__ = _import_hook
launch_id = (os.environ.get("KAGGLE_LAUNCH_ID") or "none").strip()
print(f"{_READY} launch={launch_id} | no-torch-import", flush=True)
'''

if PATCH_MANAGER_HF_GATED_DOWNLOADS:
    SITECUSTOMIZE_FILE.write_text(SITE_SOURCE.lstrip(), encoding="utf-8")
    # CORREÇÃO CENTRAL DA V4.4: o Launcher/Supervisor herda esta variável.
    os.environ["KAGGLE_HF_PATCH_DIR"] = str(SITECUSTOMIZE_DIR)
    os.environ["KAGGLE_DOWNLOAD_MIN_FREE_GIB"] = str(DOWNLOAD_MIN_FREE_GIB)
    print("✅ Patch HF gated criado e ativado para subprocessos:")
    print(SITECUSTOMIZE_FILE)
    print("KAGGLE_HF_PATCH_DIR =", os.environ["KAGGLE_HF_PATCH_DIR"])
else:
    os.environ.pop("KAGGLE_HF_PATCH_DIR", None)
    SITECUSTOMIZE_FILE.unlink(missing_ok=True)
    print("Patch HF gated desabilitado e removido.")

print("O token NÃO é gravado no patch; ele é lido do ambiente do processo.")


In [ ]:
# ============================================================
# 5) STORAGE ROUTER — V4.4
# ============================================================

import textwrap

model_subdirs = [
    "checkpoints", "diffusion_models", "unet",
    "text_encoders", "clip", "clip_vision",
    "configs", "controlnet", "vae", "vae_approx",
    "loras", "embeddings", "upscale_models", "latent_upscale_models",
    "gligen", "style_models", "hypernetworks",
    "photomaker", "audio_encoders", "model_patches",
]

for sub in model_subdirs:
    (MODELS_DIR / sub).mkdir(parents=True, exist_ok=True)

CUSTOM_NODES_DIR.mkdir(parents=True, exist_ok=True)
USER_DIR.mkdir(parents=True, exist_ok=True)

extra_yaml = COMFY_DIR / "extra_model_paths.yaml"
extra_yaml.write_text(textwrap.dedent('''
kaggle_scratch:
  base_path: /kaggle
  is_default: true

  download_model_base: temp/comfyui/models
  custom_nodes: working/ComfyUI/custom_nodes

  checkpoints: temp/comfyui/models/checkpoints

  diffusion_models: |
    temp/comfyui/models/diffusion_models
    temp/comfyui/models/unet

  text_encoders: |
    temp/comfyui/models/text_encoders
    temp/comfyui/models/clip

  clip_vision: temp/comfyui/models/clip_vision
  configs: temp/comfyui/models/configs
  controlnet: temp/comfyui/models/controlnet
  vae: temp/comfyui/models/vae
  vae_approx: temp/comfyui/models/vae_approx
  embeddings: temp/comfyui/models/embeddings
  loras: temp/comfyui/models/loras
  upscale_models: temp/comfyui/models/upscale_models
  latent_upscale_models: temp/comfyui/models/latent_upscale_models
  gligen: temp/comfyui/models/gligen
  style_models: temp/comfyui/models/style_models
  hypernetworks: temp/comfyui/models/hypernetworks
  photomaker: temp/comfyui/models/photomaker
  audio_encoders: temp/comfyui/models/audio_encoders
  model_patches: temp/comfyui/models/model_patches
''').lstrip(), encoding="utf-8")

print(extra_yaml.read_text())
print("✅ Storage router configurado.")


In [ ]:
# ============================================================
# 6) MANAGER — V4.4
# ============================================================

import configparser

if not ENABLE_MANAGER:
    print("Manager desabilitado.")
else:
    manager_dir = USER_DIR / "__manager"
    manager_dir.mkdir(parents=True, exist_ok=True)
    config_path = manager_dir / "config.ini"

    cfg = configparser.ConfigParser()
    if config_path.exists():
        cfg.read(config_path)
    if "default" not in cfg:
        cfg["default"] = {}

    cfg["default"]["network_mode"] = MANAGER_NETWORK_MODE
    cfg["default"]["security_level"] = MANAGER_SECURITY_LEVEL
    cfg["default"]["allow_git_url_install"] = str(ALLOW_GIT_URL_INSTALL).lower()
    cfg["default"]["allow_pip_install"] = str(ALLOW_PIP_INSTALL).lower()
    # Mantém o caminho torchvision, que recebe o patch HF da célula 4B.
    cfg["default"]["model_download_by_agent"] = "false"

    with config_path.open("w", encoding="utf-8") as f:
        cfg.write(f)

    pip_blacklist = manager_dir / "pip_blacklist.list"
    existing = set()
    if pip_blacklist.exists():
        existing.update(
            x.strip()
            for x in pip_blacklist.read_text(encoding="utf-8").splitlines()
            if x.strip()
        )
    existing.update({"torch", "torchvision", "torchaudio"})
    pip_blacklist.write_text("\n".join(sorted(existing)) + "\n", encoding="utf-8")

    print(config_path.read_text(encoding="utf-8"))
    print("pip blacklist:", pip_blacklist.read_text(encoding="utf-8").strip())
    if ALLOW_GIT_URL_INSTALL or ALLOW_PIP_INSTALL:
        print("⚠️ Instalação arbitrária foi habilitada explicitamente.")
    else:
        print("✅ Superfícies arbitrárias git/pip permanecem bloqueadas.")
    print("✅ Manager configurado.")


In [ ]:
# ============================================================
# 6B) V4.4 — SERVER-SIDE MODEL DOWNLOAD BRIDGE
# Intercepta "Baixar" em Modelos Ausentes e usa o Manager no servidor.
# ============================================================

import shutil
from pathlib import Path

BRIDGE_DIR = CUSTOM_NODES_DIR / "kaggle-server-downloads"
BRIDGE_WEB = BRIDGE_DIR / "web"

if not ENABLE_SERVER_SIDE_DOWNLOAD_BRIDGE:
    if BRIDGE_DIR.exists():
        shutil.rmtree(BRIDGE_DIR, ignore_errors=True)
    print("Server-side download bridge desabilitado/removido.")
else:
    BRIDGE_WEB.mkdir(parents=True, exist_ok=True)

    (BRIDGE_DIR / "__init__.py").write_text(
        '''NODE_CLASS_MAPPINGS = {}
NODE_DISPLAY_NAME_MAPPINGS = {}
WEB_DIRECTORY = "./web"
__all__ = ["NODE_CLASS_MAPPINGS", "NODE_DISPLAY_NAME_MAPPINGS", "WEB_DIRECTORY"]
''',
        encoding="utf-8",
    )

    bridge_js = r'''
import { app } from "../../scripts/app.js";
import { api } from "../../scripts/api.js";

const PREFIX = "[kaggle-server-downloads]";
const CLIENT_ID = "kaggle-server-downloads-v4.4";
let busy = false;

function notify(message, severity = "info", life = 9000) {
    try {
        if (app.extensionManager?.toast?.add) {
            app.extensionManager.toast.add({
                severity,
                summary: "Download no servidor",
                detail: message,
                life,
            });
            return;
        }
    } catch (_) {}

    const el = document.createElement("div");
    el.textContent = message;
    Object.assign(el.style, {
        position: "fixed", right: "20px", bottom: "20px", zIndex: "999999",
        maxWidth: "520px", padding: "12px 16px", borderRadius: "8px",
        fontFamily: "system-ui,sans-serif", fontSize: "13px", color: "#fff",
        background: severity === "error" ? "#7f1d1d" :
                    severity === "warn" ? "#854d0e" :
                    severity === "success" ? "#166534" : "#1e3a8a",
        boxShadow: "0 8px 24px rgba(0,0,0,.35)",
    });
    document.body.appendChild(el);
    setTimeout(() => el.remove(), life);
}

function normalizeText(value) {
    return String(value || "").trim().toLocaleLowerCase().normalize("NFD")
        .replace(/[\u0300-\u036f]/g, "");
}

function directUrl(url) {
    if (!url) return url;
    try {
        const u = new URL(url, window.location.href);
        if (u.hostname === "huggingface.co") {
            u.pathname = u.pathname.replace("/blob/", "/resolve/");
        }
        return u.toString();
    } catch (_) {
        return String(url).replace("/blob/", "/resolve/");
    }
}

function filenameFromUrl(url) {
    try {
        const u = new URL(url, window.location.href);
        const p = decodeURIComponent(u.pathname);
        return p.split("/").filter(Boolean).pop() || null;
    } catch (_) {
        return String(url || "").split("?")[0].split("/").pop() || null;
    }
}

function cleanDirectory(directory) {
    let d = String(directory || "").trim().replaceAll("\\", "/");
    d = d.replace(/^\/+/, "").replace(/^models\//, "");
    if (!d || d.includes("..") || d.includes(":")) return null;
    return d;
}

function normalizeModel(raw) {
    if (!raw) return null;
    const url = directUrl(raw.url);
    const name = raw.name || raw.filename || filenameFromUrl(url);
    const directory = cleanDirectory(raw.directory || raw.save_path || raw.type);
    if (!url || !name || !directory) return null;
    return { name, url, directory, filename: raw.filename || name };
}

function collectWorkflowModels() {
    const out = [];
    const seen = new Set();

    function add(raw) {
        const model = normalizeModel(raw);
        if (!model) return;
        const key = `${model.directory}/${model.filename}`;
        if (seen.has(key)) return;
        seen.add(key);
        out.push(model);
    }

    function walk(graph) {
        if (!graph) return;
        const nodes = graph._nodes || graph.nodes || [];
        for (const node of nodes) {
            const models = node?.properties?.models;
            if (Array.isArray(models)) for (const model of models) add(model);
            if (node?.subgraph) walk(node.subgraph);
        }
    }

    try {
        walk(app.graph);
        const root = app.graph?.rootGraph;
        if (root && root !== app.graph) walk(root);
    } catch (error) {
        console.warn(PREFIX, "Falha lendo properties.models:", error);
    }
    return out;
}

const downloadWords = [
    "baixar", "baixar tudo", "download", "download all", "download available",
    "descargar", "descargar todo", "telecharger"
];

function looksLikeDownloadButton(el) {
    const text = normalizeText(el?.textContent);
    if (downloadWords.some((word) => text === word || text.startsWith(word + " "))) return true;
    if (text.startsWith("baixar tudo (") || text.startsWith("download all (")) return true;
    return !!el?.querySelector?.("[class*='download'], [data-icon*='download'], svg[class*='download']");
}

function isInsideMissingModels(el) {
    let cursor = el;
    for (let depth = 0; depth < 9 && cursor; depth++) {
        const text = normalizeText(cursor.textContent);
        if (
            text.includes("modelos ausentes") ||
            text.includes("missing models") ||
            text.includes("modelos faltantes")
        ) {
            if (text.length < 12000) return true;
        }
        cursor = cursor.parentElement;
    }
    return false;
}

function isDownloadAll(el) {
    const text = normalizeText(el?.textContent);
    return text.startsWith("baixar tudo") || text.startsWith("download all") || text.startsWith("descargar todo");
}

function chooseSingleModel(clicked, models) {
    let cursor = clicked;
    for (let depth = 0; depth < 7 && cursor; depth++) {
        const text = cursor.textContent || "";
        const matches = models.filter((m) => text.includes(m.name));
        if (matches.length === 1) return matches[0];
        cursor = cursor.parentElement;
    }

    const anchor = clicked.closest?.("a[href]") || clicked.querySelector?.("a[href]");
    if (anchor?.href) {
        const url = directUrl(anchor.href);
        const filename = filenameFromUrl(url);
        const byUrl = models.find((m) => directUrl(m.url) === url);
        if (byUrl) return byUrl;
        const byName = models.find((m) => m.name === filename || m.filename === filename);
        if (byName) return byName;
    }
    return null;
}

function managerPayload(model, uiId) {
    return {
        name: model.name,
        url: directUrl(model.url),
        filename: model.filename || model.name,
        save_path: model.directory,
        type: model.directory,
        base: "other",
        ui_id: uiId,
        directory: model.directory,
    };
}

async function fetchText(path) {
    try {
        const response = await api.fetchApi(path, { method: "GET" });
        return { ok: response.ok, status: response.status, text: await response.text() };
    } catch (error) {
        return { ok: false, status: null, text: String(error) };
    }
}

function httpError(label, response, text = "") {
    const error = new Error(`${label} HTTP ${response.status}: ${String(text).slice(0, 300)}`);
    error.httpStatus = response.status;
    return error;
}

async function startQueue(prefix) {
    const path = `${prefix}/manager/queue/start`;
    let response = await api.fetchApi(path, { method: "POST" });
    if (response.status === 405) response = await api.fetchApi(path, { method: "GET" });
    if (!response.ok && response.status !== 201) {
        throw httpError("queue/start", response, await response.text().catch(() => ""));
    }
}

async function installV4Task(models) {
    for (const model of models) {
        const randomPart = globalThis.crypto?.randomUUID?.() || String(Math.random()).slice(2);
        const uiId = `kaggle-model-${Date.now()}-${randomPart}`;
        const payload = managerPayload(model, uiId);
        const response = await api.fetchApi("/v2/manager/queue/task", {
            method: "POST",
            headers: { "Content-Type": "application/json" },
            body: JSON.stringify({
                ui_id: uiId,
                client_id: CLIENT_ID,
                kind: "install-model",
                params: payload,
            }),
        });
        if (!response.ok) {
            throw httpError("v4 task", response, await response.text().catch(() => ""));
        }
    }
    await startQueue("/v2");
    return "v4-task";
}

async function installV2Dedicated(models) {
    for (const model of models) {
        const uiId = `kaggle-v2-${Date.now()}-${String(Math.random()).slice(2)}`;
        const response = await api.fetchApi("/v2/manager/queue/install_model", {
            method: "POST",
            headers: { "Content-Type": "application/json" },
            body: JSON.stringify(managerPayload(model, uiId)),
        });
        if (!response.ok) {
            throw httpError("v2 install_model", response, await response.text().catch(() => ""));
        }
    }
    await startQueue("/v2");
    return "v2-install_model";
}

async function installV2Batch(models) {
    const payloads = models.map((model, index) =>
        managerPayload(model, `kaggle-batch-${Date.now()}-${index}`)
    );
    const response = await api.fetchApi("/v2/manager/queue/batch", {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify({ install_model: payloads }),
    });
    if (!response.ok) {
        throw httpError("v2 batch", response, await response.text().catch(() => ""));
    }
    await startQueue("/v2");
    return "v2-batch";
}

async function installLegacy(models) {
    for (const model of models) {
        const uiId = `kaggle-legacy-${Date.now()}-${String(Math.random()).slice(2)}`;
        const response = await api.fetchApi("/manager/queue/install_model", {
            method: "POST",
            headers: { "Content-Type": "application/json" },
            body: JSON.stringify(managerPayload(model, uiId)),
        });
        if (!response.ok) {
            throw httpError("legacy install_model", response, await response.text().catch(() => ""));
        }
    }
    await startQueue("");
    return "legacy";
}

function routeMissing(error) {
    return error?.httpStatus === 404 || error?.httpStatus === 405;
}

async function managerInstall(models) {
    const version = await fetchText("/v2/manager/version");
    console.info(PREFIX, "Manager version probe:", version.status, version.text);

    try {
        return await installV4Task(models);
    } catch (error) {
        if (!routeMissing(error)) throw error;
        console.warn(PREFIX, "Unified task indisponível; tentando v2 install_model:", error);
    }

    try {
        return await installV2Dedicated(models);
    } catch (error) {
        if (!routeMissing(error)) throw error;
        console.warn(PREFIX, "v2 install_model indisponível; tentando batch:", error);
    }

    try {
        return await installV2Batch(models);
    } catch (error) {
        if (!routeMissing(error)) throw error;
        console.warn(PREFIX, "v2 batch indisponível; tentando legado:", error);
    }

    return await installLegacy(models);
}

async function onClick(event) {
    if (busy) return;
    if (event.button !== undefined && event.button !== 0) return;

    const button = event.target?.closest?.("button, a");
    if (!button || !looksLikeDownloadButton(button) || !isInsideMissingModels(button)) return;

    event.preventDefault();
    event.stopPropagation();
    event.stopImmediatePropagation();

    const models = collectWorkflowModels();
    if (!models.length) {
        notify(
            "O download local foi bloqueado, mas este workflow não expôs properties.models[]. Use Manager → Model Manager.",
            "error", 13000
        );
        return;
    }

    let queue = models;
    if (!isDownloadAll(button)) {
        const single = chooseSingleModel(button, models);
        if (!single) {
            notify(
                "Não consegui identificar com segurança o arquivo desta linha. Use 'Baixar tudo' ou o Model Manager.",
                "warn", 13000
            );
            return;
        }
        queue = [single];
    }

    busy = true;
    notify(`Enviando ${queue.length} arquivo${queue.length === 1 ? "" : "s"} ao Manager no Kaggle…`);

    try {
        const dialect = await managerInstall(queue);
        notify(
            `✓ ${queue.length} download${queue.length === 1 ? "" : "s"} iniciado${queue.length === 1 ? "" : "s"} no servidor (${dialect}). Destino: /kaggle/temp/comfyui/models.`,
            "success", 13000
        );
        console.info(PREFIX, "downloads enfileirados:", dialect, queue);
    } catch (error) {
        console.error(PREFIX, error);
        notify(
            `Falha ao enviar download ao Manager: ${String(error).slice(0, 350)}. O download local permaneceu bloqueado.`,
            "error", 16000
        );
    } finally {
        busy = false;
    }
}

app.registerExtension({
    name: "kaggle.server-side-downloads.v4.4",
    init() {
        document.addEventListener("click", onClick, { capture: true });
        console.info(PREFIX, "V4.4 ativo — downloads de Modelos Ausentes serão server-side");
    },
});
'''

    (BRIDGE_WEB / "server-side-downloads.js").write_text(bridge_js, encoding="utf-8")
    print("✅ kaggle-server-downloads V4.4 instalado.")
    print("Python :", BRIDGE_DIR / "__init__.py")
    print("JS     :", BRIDGE_WEB / "server-side-downloads.js")
    print("O bridge será carregado na próxima inicialização/restart do ComfyUI.")


In [ ]:
# ============================================================
# 7) CLOUDFLARED — download atômico
# ============================================================

import os
import subprocess

CLOUDFLARED_BIN.parent.mkdir(parents=True, exist_ok=True)

needs_download = REFRESH_CLOUDFLARED or not CLOUDFLARED_BIN.exists()

if needs_download:
    if CLOUDFLARED_VERSION:
        url = (
            "https://github.com/cloudflare/cloudflared/releases/download/"
            f"{CLOUDFLARED_VERSION}/cloudflared-linux-amd64"
        )
    else:
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"

    tmp = CLOUDFLARED_BIN.with_suffix(".download")
    tmp.unlink(missing_ok=True)
    print("Baixando cloudflared:", url)
    subprocess.run([
        "wget", "-q", "--show-progress", "-O", str(tmp), url
    ], check=True)
    tmp.chmod(0o755)

    test = subprocess.run(
        [str(tmp), "--version"],
        text=True,
        capture_output=True,
    )
    if test.returncode != 0:
        tmp.unlink(missing_ok=True)
        raise RuntimeError("Binário cloudflared baixado falhou em --version.")

    os.replace(tmp, CLOUDFLARED_BIN)
    CLOUDFLARED_BIN.chmod(0o755)

version = subprocess.run(
    [str(CLOUDFLARED_BIN), "--version"],
    text=True,
    capture_output=True,
    check=False,
)
print(version.stdout.strip() or version.stderr.strip())
if version.returncode != 0:
    raise RuntimeError("cloudflared existente não é executável/válido.")
print("✅ cloudflared pronto.")


In [ ]:
# ============================================================
# 7B) TESTAR O MESMO DOWNLOADER USADO PELO MANAGER — V4.4
# ============================================================

import os
import subprocess
import sys
from pathlib import Path

if not PATCH_MANAGER_HF_GATED_DOWNLOADS:
    print("Patch HF gated desabilitado.")
else:
    token = (os.environ.get("HF_TOKEN") or "").strip()
    if not token:
        raise RuntimeError("HF_TOKEN ausente. Execute 1B e 1C.")

    if os.environ.get("KAGGLE_HF_PATCH_DIR") != str(SITECUSTOMIZE_DIR):
        raise RuntimeError("KAGGLE_HF_PATCH_DIR não aponta para o patch criado em 4B.")

    test_root = Path("/kaggle/temp/manager-hf-auth-test")
    test_root.mkdir(parents=True, exist_ok=True)
    target = test_root / "ltx25-gated-test.gitattributes"
    partial = Path(str(target) + ".part")
    target.unlink(missing_ok=True)
    partial.unlink(missing_ok=True)

    test_url = (
        "https://huggingface.co/"
        f"{HF_GATED_TEST_REPO}/resolve/main/{HF_GATED_TEST_FILE}"
    )

    child_code = r'''
import os
from torchvision.datasets.utils import download_url

download_url(
    os.environ["TEST_HF_URL"],
    os.environ["TEST_HF_ROOT"],
    filename="ltx25-gated-test.gitattributes",
)
print("MANAGER_STYLE_HF_DOWNLOAD_OK")
'''

    env = os.environ.copy()
    old_pp = env.get("PYTHONPATH", "")
    patch_dir = str(SITECUSTOMIZE_DIR)
    env["PYTHONPATH"] = patch_dir + ((os.pathsep + old_pp) if old_pp else "")
    env["TEST_HF_URL"] = test_url
    env["TEST_HF_ROOT"] = str(test_root)

    proc = subprocess.run(
        [sys.executable, "-c", child_code],
        env=env,
        text=True,
        capture_output=True,
    )

    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError("O teste do downloader estilo Manager falhou.")

    if HF_PATCH_SIGNATURE not in proc.stdout:
        print(proc.stderr)
        raise RuntimeError(
            "O download pode até ter terminado, mas o sitecustomize V4.4 não confirmou carga."
        )

    if "no-torch-import" not in proc.stdout:
        raise RuntimeError(
            "O patch HF foi carregado, mas o marcador no-torch-import não apareceu. "
            "O sitecustomize pode estar importando Torch cedo demais."
        )

    if (
        "MANAGER_STYLE_HF_DOWNLOAD_OK" not in proc.stdout
        or not target.exists()
        or target.stat().st_size == 0
    ):
        raise RuntimeError("O subprocesso terminou, mas o arquivo autenticado não foi criado.")

    print("=" * 78)
    print("✅ MANAGER + HF GATED PATCH V4.4 VALIDADO")
    print("=" * 78)
    print("Bytes:", target.stat().st_size)


In [ ]:
# ============================================================
# 8) LAUNCHER ROBUSTO — V4.4
# ComfyUI + supervisor + process groups + tunnel + probes
# ============================================================

import os
import re
import signal
import socket
import subprocess
import sys
import textwrap
import time
import urllib.error
import urllib.request
from pathlib import Path

if STOP_FLAG.exists():
    STOP_FLAG.unlink()

supervisor_script = RUNTIME_DIR / "comfyui_supervisor.py"


def pid_alive(pid):
    if not pid:
        return False
    try:
        os.kill(pid, 0)
        return True
    except Exception:
        return False


def read_pid(path):
    try:
        return int(path.read_text().strip())
    except Exception:
        return None


def proc_cmdline(pid):
    try:
        raw = Path(f"/proc/{pid}/cmdline").read_bytes()
        return raw.replace(b"\x00", b" ").decode(errors="replace")
    except Exception:
        return ""


def terminate_pidfile_group(path, label, expected_fragment=None, wait=3):
    pid = read_pid(path)
    if not pid:
        path.unlink(missing_ok=True)
        return

    if not pid_alive(pid):
        path.unlink(missing_ok=True)
        return

    cmdline = proc_cmdline(pid)
    if expected_fragment and expected_fragment not in cmdline:
        print(
            f"⚠️ {label}: PID file aponta para PID {pid}, mas a identidade não confere; "
            "não vou matar um processo possivelmente reutilizado."
        )
        path.unlink(missing_ok=True)
        return

    print(f"Encerrando {label} anterior PID {pid} / PGID controlado")
    try:
        pgid = os.getpgid(pid)
    except Exception:
        pgid = None

    try:
        if pgid:
            os.killpg(pgid, signal.SIGTERM)
        else:
            os.kill(pid, signal.SIGTERM)
    except ProcessLookupError:
        pass

    deadline = time.time() + wait
    while pid_alive(pid) and time.time() < deadline:
        time.sleep(0.1)

    if pid_alive(pid):
        try:
            if pgid:
                os.killpg(pgid, signal.SIGKILL)
            else:
                os.kill(pid, signal.SIGKILL)
        except ProcessLookupError:
            pass

    path.unlink(missing_ok=True)


def local_port_ok():
    try:
        with socket.create_connection((LISTEN_HOST, PORT), timeout=2):
            return True
    except OSError:
        return False


def terminate_orphan_comfy():
    target_main = str(COMFY_DIR / "main.py")
    victims = []
    for entry in Path("/proc").iterdir():
        if not entry.name.isdigit():
            continue
        pid = int(entry.name)
        if pid == os.getpid():
            continue
        cmdline = proc_cmdline(pid)
        if target_main in cmdline and "--port" in cmdline and str(PORT) in cmdline:
            victims.append(pid)

    for pid in victims:
        print("Encerrando ComfyUI órfão PID", pid)
        try:
            os.kill(pid, signal.SIGTERM)
        except ProcessLookupError:
            pass

    if victims:
        time.sleep(2)
        for pid in victims:
            if pid_alive(pid):
                try:
                    os.kill(pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
    return victims


def http_probe(url, timeout=PUBLIC_PROBE_TIMEOUT):
    headers = {
        "User-Agent": "Mozilla/5.0 ComfyUI-Kaggle-Healthcheck/4.3",
        "Accept": "text/html,application/json,*/*",
        "Cache-Control": "no-cache",
        "Pragma": "no-cache",
    }
    req = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            body = r.read(800).decode("utf-8", errors="replace")
            return {"ok": 200 <= r.status < 400, "status": r.status, "body": body, "error": None}
    except urllib.error.HTTPError as e:
        body = e.read(800).decode("utf-8", errors="replace")
        return {"ok": False, "status": e.code, "body": body, "error": repr(e)}
    except Exception as e:
        return {"ok": False, "status": None, "body": "", "error": repr(e)}


# Limpa instâncias anteriores de forma segura.
terminate_pidfile_group(CLOUDFLARE_PID, "Cloudflare", "cloudflared", wait=2)
terminate_pidfile_group(SUPERVISOR_PID, "Supervisor", "comfyui_supervisor.py", wait=3)
COMFY_PID.unlink(missing_ok=True)

# Caso uma V4.2 antiga tenha deixado apenas o filho vivo.
if local_port_ok():
    victims = terminate_orphan_comfy()
    time.sleep(1)
    if local_port_ok():
        raise RuntimeError(
            f"A porta {PORT} continua ocupada após a limpeza. "
            f"Processos ComfyUI órfãos encontrados: {victims}."
        )

LAUNCH_ID = str(time.time_ns())
os.environ["KAGGLE_LAUNCH_ID"] = LAUNCH_ID
LAUNCH_ID_FILE.write_text(LAUNCH_ID, encoding="utf-8")
CURRENT_HF_PATCH_MARKER = f"{HF_PATCH_SIGNATURE} launch={LAUNCH_ID}"

if PATCH_MANAGER_HF_GATED_DOWNLOADS:
    if not SITECUSTOMIZE_FILE.exists():
        raise RuntimeError("Patch HF gated habilitado, mas sitecustomize.py não existe. Execute 4B.")
    if os.environ.get("KAGGLE_HF_PATCH_DIR") != str(SITECUSTOMIZE_DIR):
        raise RuntimeError("KAGGLE_HF_PATCH_DIR não foi ativado. Execute novamente 4B.")

comfy_cmd = [
    sys.executable, "-u", str(COMFY_DIR / "main.py"),
    "--listen", LISTEN_HOST,
    "--port", str(PORT),
    "--disable-auto-launch",
    "--cuda-device", CUDA_DEVICE_IDS,
    "--default-device", str(DEFAULT_GPU),
    "--input-directory", str(INPUT_DIR),
    "--output-directory", str(OUTPUT_DIR),
    "--temp-directory", str(TEMP_DIR),
    "--user-directory", str(USER_DIR),
    "--extra-model-paths-config", str(extra_yaml),
    "--preview-method", "auto",
]

if ENABLE_REMOTE_CORS:
    comfy_cmd.extend(["--enable-cors-header", CORS_ORIGIN])
if ENABLE_ASSETS:
    comfy_cmd.append("--enable-assets")
if ENABLE_MANAGER:
    comfy_cmd.append("--enable-manager")

if CUDA_MALLOC_MODE == "async":
    comfy_cmd.append("--cuda-malloc")
elif CUDA_MALLOC_MODE == "native":
    comfy_cmd.append("--disable-cuda-malloc")
elif CUDA_MALLOC_MODE != "auto":
    raise ValueError(f"CUDA_MALLOC_MODE inválido: {CUDA_MALLOC_MODE}")
if USE_LEGACY_MANAGER_UI:
    comfy_cmd.append("--enable-manager-legacy-ui")
comfy_cmd.extend(EXTRA_COMFY_ARGS)

print("Comando ComfyUI:")
print(" ".join(map(str, comfy_cmd)))

supervisor_template = r'''
import ctypes
import os
import signal
import subprocess
import time
from pathlib import Path

cmd = __CMD__
cwd = __CWD__
stop_flag = Path(__STOP_FLAG__)
log_path = Path(__COMFY_LOG__)
sup_log_path = Path(__SUP_LOG__)
child_pid_path = Path(__CHILD_PID__)

stopping = False
current_proc = None

def _signal_stop(signum, frame):
    global stopping
    stopping = True

signal.signal(signal.SIGTERM, _signal_stop)
signal.signal(signal.SIGINT, _signal_stop)

def _pdeathsig():
    # Linux/Kaggle: se o supervisor morrer abruptamente, pede SIGTERM ao filho.
    try:
        libc = ctypes.CDLL(None)
        libc.prctl(1, signal.SIGTERM)  # PR_SET_PDEATHSIG = 1
    except Exception:
        pass

def _terminate_child(proc, grace=8):
    if proc is None or proc.poll() is not None:
        return
    try:
        proc.terminate()
    except ProcessLookupError:
        return
    deadline = time.time() + grace
    while proc.poll() is None and time.time() < deadline:
        time.sleep(0.1)
    if proc.poll() is None:
        try:
            proc.kill()
        except ProcessLookupError:
            pass

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["TOKENIZERS_PARALLELISM"] = "false"

patch_dir = (os.environ.get("KAGGLE_HF_PATCH_DIR") or "").strip()
if patch_dir:
    old_pp = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = patch_dir + ((os.pathsep + old_pp) if old_pp else "")

if os.environ.get("HF_TOKEN"):
    env["HF_TOKEN"] = os.environ["HF_TOKEN"]
    env["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
    env["HUGGINGFACE_TOKEN"] = os.environ["HF_TOKEN"]

with sup_log_path.open("a", buffering=1) as slog:
    slog.write("\n=== supervisor V4.4 started ===\n")
    slog.write("command: " + " ".join(cmd) + "\n")
    slog.write("PYTHONPATH patch: " + (patch_dir or "OFF") + "\n")

    try:
        while not stop_flag.exists() and not stopping:
            with log_path.open("a", buffering=1) as log:
                log.write("\n\n=== ComfyUI process start V4.4 ===\n")
                current_proc = subprocess.Popen(
                    cmd,
                    cwd=cwd,
                    env=env,
                    stdout=log,
                    stderr=subprocess.STDOUT,
                    start_new_session=False,
                    preexec_fn=_pdeathsig,
                )
                child_pid_path.write_text(str(current_proc.pid))
                slog.write(f"ComfyUI PID {current_proc.pid}\n")

                while current_proc.poll() is None:
                    if stopping or stop_flag.exists():
                        _terminate_child(current_proc)
                        break
                    time.sleep(0.5)

                rc = current_proc.wait()
                slog.write(f"ComfyUI exited rc={rc}\n")
                child_pid_path.unlink(missing_ok=True)
                current_proc = None

            if stopping or stop_flag.exists():
                break

            slog.write("Restarting ComfyUI in 3 seconds...\n")
            for _ in range(30):
                if stopping or stop_flag.exists():
                    break
                time.sleep(0.1)
    finally:
        _terminate_child(current_proc)
        child_pid_path.unlink(missing_ok=True)
        slog.write("=== supervisor V4.4 stopped ===\n")
'''

supervisor_source = supervisor_template
supervisor_source = supervisor_source.replace("__CMD__", repr(comfy_cmd))
supervisor_source = supervisor_source.replace("__CWD__", repr(str(COMFY_DIR)))
supervisor_source = supervisor_source.replace("__STOP_FLAG__", repr(str(STOP_FLAG)))
supervisor_source = supervisor_source.replace("__COMFY_LOG__", repr(str(COMFY_LOG)))
supervisor_source = supervisor_source.replace("__SUP_LOG__", repr(str(SUPERVISOR_LOG)))
supervisor_source = supervisor_source.replace("__CHILD_PID__", repr(str(COMFY_PID)))

# Valida o script gerado ANTES de executar.
compile(supervisor_source, str(supervisor_script), "exec")
supervisor_script.write_text(supervisor_source.lstrip(), encoding="utf-8")

sup_handle = SUPERVISOR_LOG.open("a")
supervisor = subprocess.Popen(
    [sys.executable, "-u", str(supervisor_script)],
    stdout=sup_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
sup_handle.close()
SUPERVISOR_PID.write_text(str(supervisor.pid))

# Não aceita apenas "porta aberta": exige filho atual + HTTP saudável.
deadline = time.time() + 240
local_root = None
while time.time() < deadline:
    child_pid = read_pid(COMFY_PID)
    child_ok = pid_alive(child_pid)
    if child_ok:
        local_root = http_probe(f"http://{LISTEN_HOST}:{PORT}/", timeout=5)
        if local_root["ok"]:
            break
    if supervisor.poll() is not None:
        break
    time.sleep(2)

child_pid = read_pid(COMFY_PID)
if not (pid_alive(child_pid) and local_root and local_root["ok"]):
    print("\n--- ComfyUI log ---")
    if COMFY_LOG.exists():
        print("\n".join(COMFY_LOG.read_text(errors="replace").splitlines()[-180:]))
    raise RuntimeError("ComfyUI V4.4 não ficou saudável com um filho atual do supervisor.")

local_stats = http_probe(f"http://{LISTEN_HOST}:{PORT}/system_stats", timeout=10)
print("\nLocal /            :", local_root["status"], "OK" if local_root["ok"] else "FAIL")
print("Local /system_stats:", local_stats["status"], "OK" if local_stats["ok"] else "FAIL")
print("Supervisor PID     :", supervisor.pid)
print("ComfyUI PID        :", child_pid)

if PATCH_MANAGER_HF_GATED_DOWNLOADS:
    patch_deadline = time.time() + 20
    patch_seen = False
    while time.time() < patch_deadline:
        if COMFY_LOG.exists() and CURRENT_HF_PATCH_MARKER in COMFY_LOG.read_text(errors="replace"):
            patch_seen = True
            break
        time.sleep(0.5)
    if not patch_seen:
        raise RuntimeError(
            "ComfyUI abriu, mas o log NÃO confirmou o patch HF V4.4. "
            "O Launcher foi interrompido para evitar falso positivo."
        )
    print("HF patch no processo: ✅ confirmado no log")

print("✅ ComfyUI local saudável.")


def create_tunnel_once(attempt):
    terminate_pidfile_group(CLOUDFLARE_PID, "Cloudflare", "cloudflared", wait=1)
    log_path = RUNTIME_DIR / f"cloudflared-attempt-{attempt}.log"
    log_path.write_text("", encoding="utf-8")
    handle = log_path.open("a")
    proc = subprocess.Popen(
        [
            str(CLOUDFLARED_BIN), "tunnel",
            "--url", f"http://{LISTEN_HOST}:{PORT}",
            "--no-autoupdate",
        ],
        stdout=handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    handle.close()
    CLOUDFLARE_PID.write_text(str(proc.pid))

    pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
    url = None
    deadline = time.time() + TUNNEL_START_TIMEOUT
    while time.time() < deadline:
        text = log_path.read_text(errors="replace")
        m = pattern.search(text)
        if m:
            url = m.group(0)
            break
        if proc.poll() is not None:
            break
        time.sleep(1)
    return proc, url, log_path

PUBLIC_URL = None
PUBLIC_RESULT = None

for attempt in range(1, MAX_TUNNEL_ATTEMPTS + 1):
    print(f"\nCloudflare attempt {attempt}/{MAX_TUNNEL_ATTEMPTS}")
    cf_proc, candidate, cf_log = create_tunnel_once(attempt)
    if not candidate:
        print("⚠️ URL não encontrada.")
        print(cf_log.read_text(errors="replace")[-3000:])
        terminate_pidfile_group(CLOUDFLARE_PID, "Cloudflare", "cloudflared", wait=1)
        continue

    print("URL candidata:", candidate)
    result = None
    stats_result = None
    for probe_i in range(1, PUBLIC_PROBE_RETRIES + 1):
        result = http_probe(candidate + f"/?probe={int(time.time())}")
        stats_result = http_probe(candidate + f"/system_stats?probe={int(time.time())}")
        print(
            f"  probe {probe_i}/{PUBLIC_PROBE_RETRIES}: "
            f"/={result['status']} /system_stats={stats_result['status']}"
        )
        if result["ok"] and stats_result["ok"]:
            break
        time.sleep(PUBLIC_PROBE_DELAY)

    if result and stats_result and result["ok"] and stats_result["ok"]:
        PUBLIC_URL = candidate
        PUBLIC_RESULT = {"root": result, "system_stats": stats_result}
        break

    print("⚠️ Túnel não passou no teste público.")
    if result:
        print("Status:", result["status"])
        if result["body"]:
            print("Body:", result["body"][:500])
    terminate_pidfile_group(CLOUDFLARE_PID, "Cloudflare", "cloudflared", wait=1)

if not PUBLIC_URL:
    print("\n--- Últimas linhas do ComfyUI ---")
    if COMFY_LOG.exists():
        print("\n".join(COMFY_LOG.read_text(errors="replace").splitlines()[-100:]))
    raise RuntimeError("Nenhum Quick Tunnel passou nos testes HTTP públicos.")

cache_buster = int(time.time())
ACCESS_URL = f"{PUBLIC_URL}/?v={cache_buster}"
PUBLIC_URL_FILE.write_text(PUBLIC_URL)
ACCESS_URL_FILE.write_text(ACCESS_URL)

print("\n" + "=" * 78)
print("✅ COMFYUI KAGGLE V4.4 ONLINE — PUBLIC ACCESS VALIDATED")
print("=" * 78)
for g in GPU_ROWS:
    print(f"GPU {g['index']}: {g['name']} | {g['memory_mib']/1024:.2f} GiB | SM {g['compute_cap']}")
print()
print("CUDA devices :", CUDA_DEVICE_IDS)
print("Default GPU  :", DEFAULT_GPU)
print("CORS remoto  :", CORS_ORIGIN if ENABLE_REMOTE_CORS else "OFF")
print("Manager      :", "ON" if ENABLE_MANAGER else "OFF")
print("Assets       :", "ON" if ENABLE_ASSETS else "OFF")
print("DL Bridge    :", "ON" if ENABLE_SERVER_SIDE_DOWNLOAD_BRIDGE else "OFF")
print("HF Gated DL  :", "PATCHED+VERIFIED" if PATCH_MANAGER_HF_GATED_DOWNLOADS else "OFF")
print("Models       :", MODELS_DIR)
print("Custom Nodes :", CUSTOM_NODES_DIR)
print("Supervisor   :", supervisor.pid)
print("ComfyUI PID  :", read_pid(COMFY_PID))
print()
print("PUBLIC URL:")
print(PUBLIC_URL)
print()
print("ACCESS URL:")
print(ACCESS_URL)
print("=" * 78)
print("⚠️ Quick Tunnel é público/temporário. Não compartilhe a URL desnecessariamente.")


In [ ]:
# ============================================================
# 8B) VALIDAR MANAGER + BRIDGE + PATCH — V4.4
# ============================================================

import os
import subprocess
import urllib.error
import urllib.request


def get_http(url, timeout=15):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "ComfyUI-Kaggle-V4.4-Validator"})
        with urllib.request.urlopen(req, timeout=timeout) as response:
            return response.status, response.read(2500).decode("utf-8", errors="replace")
    except urllib.error.HTTPError as e:
        return e.code, e.read(2500).decode("utf-8", errors="replace")
    except Exception as e:
        return None, repr(e)

base = f"http://{LISTEN_HOST}:{PORT}"
checks = {
    "ComfyUI root": "/",
    "Manager v4 version": "/v2/manager/version",
    "Manager queue status": "/v2/manager/queue/status",
}
if ENABLE_SERVER_SIDE_DOWNLOAD_BRIDGE:
    checks["Server download bridge"] = "/extensions/kaggle-server-downloads/server-side-downloads.js"

print("=" * 78)
print("V4.4 VALIDATION")
print("=" * 78)

BRIDGE_OK = not ENABLE_SERVER_SIDE_DOWNLOAD_BRIDGE
MANAGER_OK = not ENABLE_MANAGER

for name, path in checks.items():
    status, body = get_http(base + path)
    ok = status is not None and 200 <= status < 400
    print(f"{name:28} => {status} {'✅' if ok else '⚠️'}")
    if name == "Server download bridge":
        BRIDGE_OK = ok and "kaggle.server-side-downloads.v4.4" in body
    if name == "Manager v4 version":
        MANAGER_OK = ok

sup_pid = read_pid(SUPERVISOR_PID)
child_pid = read_pid(COMFY_PID)
print(f"{'Supervisor PID':28} => {sup_pid} {'✅' if pid_alive(sup_pid) else '❌'}")
print(f"{'ComfyUI PID':28} => {child_pid} {'✅' if pid_alive(child_pid) else '❌'}")

PATCH_OK = not PATCH_MANAGER_HF_GATED_DOWNLOADS
if PATCH_MANAGER_HF_GATED_DOWNLOADS:
    launch_id = LAUNCH_ID_FILE.read_text().strip() if LAUNCH_ID_FILE.exists() else ""
    current_marker = f"{HF_PATCH_SIGNATURE} launch={launch_id}" if launch_id else ""
    PATCH_OK = (
        SITECUSTOMIZE_FILE.exists()
        and os.environ.get("KAGGLE_HF_PATCH_DIR") == str(SITECUSTOMIZE_DIR)
        and COMFY_LOG.exists()
        and bool(current_marker)
        and current_marker in COMFY_LOG.read_text(errors="replace")
    )
print(f"{'HF patch no processo':28} => {'✅' if PATCH_OK else '❌'}")
print(f"{'Bridge carregado':28} => {'✅' if BRIDGE_OK else '❌'}")
print(f"{'Manager API':28} => {'✅' if MANAGER_OK else '⚠️'}")

commit = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"], cwd=COMFY_DIR,
    text=True, capture_output=True, check=False,
).stdout.strip()
print(f"{'ComfyUI commit':28} => {commit or 'n/a'}")

if not (PATCH_OK and BRIDGE_OK and MANAGER_OK):
    print("\n⚠️ Uma ou mais validações falharam. Execute a célula 9 de diagnóstico.")
else:
    print("\n✅ Camada V4.4 validada.")

if ENABLE_SERVER_SIDE_DOWNLOAD_BRIDGE:
    print("\nApós abrir a interface, faça um HARD REFRESH uma vez:")
    print("Windows/Linux: Ctrl + Shift + R")
    print("Mac: Cmd + Shift + R")


In [ ]:
# ============================================================
# 9) DIAGNÓSTICO COMPLETO — V4.4
# Execute se a UI/download apresentar qualquer erro.
# ============================================================

import os
import shutil
import socket
import subprocess
import urllib.error
import urllib.request
from pathlib import Path


def diag_pid(path):
    pid = read_pid(path)
    return pid_alive(pid), pid


def diag_http(url):
    headers = {
        "User-Agent": "Mozilla/5.0 ComfyUI-Kaggle-Diagnostic/4.3",
        "Cache-Control": "no-cache",
    }
    try:
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=15) as r:
            return r.status, r.read(500).decode(errors="replace")
    except urllib.error.HTTPError as e:
        return e.code, e.read(500).decode(errors="replace")
    except Exception as e:
        return None, repr(e)

print("=" * 78)
print("CUDA ALLOCATOR")
print("=" * 78)
print("Notebook mode              :", CUDA_MALLOC_MODE)
print("PYTORCH_CUDA_ALLOC_CONF    :", os.environ.get("PYTORCH_CUDA_ALLOC_CONF") or "<unset>")
print("PYTORCH_ALLOC_CONF        :", os.environ.get("PYTORCH_ALLOC_CONF") or "<unset>")
print("\n" + "=" * 78)
print("PROCESSOS")
print("=" * 78)
for name, path in [
    ("Supervisor", SUPERVISOR_PID),
    ("ComfyUI", COMFY_PID),
    ("Cloudflare", CLOUDFLARE_PID),
]:
    ok, pid = diag_pid(path)
    print(f"{name:12}: {'OK' if ok else 'DOWN'}", pid or "")
    if pid:
        print(" " * 14 + proc_cmdline(pid)[:500])

print("\n" + "=" * 78)
print("LOCAL")
print("=" * 78)
for endpoint in ["/", "/system_stats"]:
    status, body = diag_http(f"http://{LISTEN_HOST}:{PORT}{endpoint}")
    print(endpoint, "=>", status)
    if status is None or (status and status >= 400):
        print(body[:500])

print("\n" + "=" * 78)
print("PUBLIC")
print("=" * 78)
if PUBLIC_URL_FILE.exists():
    pub = PUBLIC_URL_FILE.read_text().strip()
    for endpoint in ["/", "/system_stats"]:
        url = pub.rstrip("/") + endpoint
        status, body = diag_http(url)
        print(url, "=>", status)
        if status is None or (status and status >= 400):
            print(body[:500])
else:
    print("PUBLIC_URL_FILE ausente.")

print("\n" + "=" * 78)
print("NVIDIA-SMI")
print("=" * 78)
subprocess.run(["nvidia-smi"], check=False)

print("\n" + "=" * 78)
print("COMFYUI LOG — últimas 160 linhas")
print("=" * 78)
if COMFY_LOG.exists():
    print("\n".join(COMFY_LOG.read_text(errors="replace").splitlines()[-160:]))

print("\n" + "=" * 78)
print("SUPERVISOR LOG — últimas 100 linhas")
print("=" * 78)
if SUPERVISOR_LOG.exists():
    print("\n".join(SUPERVISOR_LOG.read_text(errors="replace").splitlines()[-100:]))

print("\n" + "=" * 78)
print("SERVER-SIDE DOWNLOAD BRIDGE")
print("=" * 78)
if ENABLE_SERVER_SIDE_DOWNLOAD_BRIDGE:
    bridge_url = f"http://{LISTEN_HOST}:{PORT}/extensions/kaggle-server-downloads/server-side-downloads.js"
    status, body = diag_http(bridge_url)
    print("Bridge JS =>", status)
    if status == 200:
        print("Bridge signature:", "OK" if "kaggle.server-side-downloads.v4.4" in body else "UNKNOWN")
else:
    print("Bridge desabilitado por configuração.")

print("\n" + "=" * 78)
print("MODEL STORAGE")
print("=" * 78)
du = shutil.disk_usage(MODELS_DIR)
print("Models dir :", MODELS_DIR)
print("Free GiB   :", f"{du.free / 1024**3:.1f}")
for sub in [
    "checkpoints", "loras", "text_encoders", "latent_upscale_models",
    "vae", "vae_approx", "diffusion_models",
]:
    p = MODELS_DIR / sub
    files = [x for x in p.glob("*") if x.is_file()] if p.exists() else []
    size = sum(x.stat().st_size for x in files)
    print(f"{sub:24} files={len(files):3d} size={size / 1024**3:7.2f} GiB")

print("\n" + "=" * 78)
print("HF GATED MANAGER PATCH")
print("=" * 78)
print("Sitecustomize :", SITECUSTOMIZE_FILE)
print("Existe        :", SITECUSTOMIZE_FILE.exists())
print("Patch env     :", os.environ.get("KAGGLE_HF_PATCH_DIR") or "OFF")
print("HF_TOKEN env  :", "✅ presente" if bool(os.environ.get("HF_TOKEN")) else "❌ ausente")
launch_id = LAUNCH_ID_FILE.read_text().strip() if LAUNCH_ID_FILE.exists() else ""
current_marker = f"{HF_PATCH_SIGNATURE} launch={launch_id}" if launch_id else ""
print(
    "Process log   :",
    "✅ V4.4 carregado no launch atual"
    if COMFY_LOG.exists() and current_marker and current_marker in COMFY_LOG.read_text(errors="replace")
    else "❌ assinatura do launch atual ausente"
)
print("Launch ID      :", launch_id or "ausente")
_patch_test = Path("/kaggle/temp/manager-hf-auth-test/ltx25-gated-test.gitattributes")
print("Manager test  :", "✅ OK" if _patch_test.exists() else "⚠️ não executado")

print("\n" + "=" * 78)
print("MANAGER CONFIG")
print("=" * 78)
config_path = USER_DIR / "__manager" / "config.ini"
if config_path.exists():
    print(config_path.read_text(errors="replace"))
else:
    print("config.ini ausente")


## Keep-alive resiliente — V4.4

A próxima célula fica executando continuamente.

Ela testa a instância local, o Supervisor, o processo do túnel e periodicamente a URL pública. Se o túnel falhar repetidamente, cria um novo Quick Tunnel e atualiza `public_url.txt`/`access_url.txt`.

Interromper a célula com `KeyboardInterrupt` **não** encerra ComfyUI; para isso use `ACTION = "stop_all"` na célula de controle.


In [ ]:
# ============================================================
# 10) KEEP-ALIVE + WATCHDOG DE TÚNEL — V4.4
# DEIXE ESTA CÉLULA RODANDO.
# ============================================================

import datetime
import os
import re
import signal
import socket
import subprocess
import time
import urllib.error
import urllib.request


def watchdog_pid_alive(path):
    pid = read_pid(path)
    return pid_alive(pid), pid


def watchdog_local_ok():
    ok, child = watchdog_pid_alive(COMFY_PID)
    if not ok:
        return False
    try:
        with socket.create_connection((LISTEN_HOST, PORT), timeout=3):
            pass
        req = urllib.request.Request(
            f"http://{LISTEN_HOST}:{PORT}/system_stats",
            headers={"User-Agent": "ComfyUI-Kaggle-Watchdog/4.3"},
        )
        with urllib.request.urlopen(req, timeout=8) as r:
            return 200 <= r.status < 400
    except Exception:
        return False


def watchdog_http(url):
    headers = {
        "User-Agent": "Mozilla/5.0 ComfyUI-Kaggle-Watchdog/4.3",
        "Cache-Control": "no-cache",
    }
    try:
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=15) as r:
            return 200 <= r.status < 400, r.status
    except urllib.error.HTTPError as e:
        return False, e.code
    except Exception:
        return False, None


def watchdog_stop_cloudflare():
    terminate_pidfile_group(CLOUDFLARE_PID, "Cloudflare", "cloudflared", wait=2)


def watchdog_new_tunnel():
    watchdog_stop_cloudflare()
    log = RUNTIME_DIR / "cloudflared-watchdog.log"
    log.write_text("", encoding="utf-8")
    handle = log.open("a")
    proc = subprocess.Popen(
        [
            str(CLOUDFLARED_BIN), "tunnel",
            "--url", f"http://{LISTEN_HOST}:{PORT}",
            "--no-autoupdate",
        ],
        stdout=handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    handle.close()
    CLOUDFLARE_PID.write_text(str(proc.pid))

    pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
    deadline = time.time() + TUNNEL_START_TIMEOUT
    while time.time() < deadline:
        text = log.read_text(errors="replace")
        m = pattern.search(text)
        if m:
            candidate = m.group(0)
            for _ in range(PUBLIC_PROBE_RETRIES):
                ok, status = watchdog_http(candidate + f"/?watchdog={int(time.time())}")
                if ok:
                    access = candidate + f"/?v={int(time.time())}"
                    PUBLIC_URL_FILE.write_text(candidate)
                    ACCESS_URL_FILE.write_text(access)
                    return True, candidate, access, status
                time.sleep(PUBLIC_PROBE_DELAY)
            break
        if proc.poll() is not None:
            break
        time.sleep(1)

    watchdog_stop_cloudflare()
    return False, None, None, None

print("=" * 78)
print("KEEP-ALIVE V4.4 ATIVO")
print("=" * 78)
if ACCESS_URL_FILE.exists():
    print("ACCESS URL:")
    print(ACCESS_URL_FILE.read_text().strip())

public_failures = 0
heartbeat_count = 0

try:
    while True:
        heartbeat_count += 1
        local_ok = watchdog_local_ok()
        sup_ok, sup_pid = watchdog_pid_alive(SUPERVISOR_PID)
        child_ok, child_pid = watchdog_pid_alive(COMFY_PID)
        cf_ok, cf_pid = watchdog_pid_alive(CLOUDFLARE_PID)

        public_ok = None
        public_status = None
        if heartbeat_count % PUBLIC_CHECK_EVERY == 0 and PUBLIC_URL_FILE.exists():
            pub = PUBLIC_URL_FILE.read_text().strip()
            public_ok, public_status = watchdog_http(pub + f"/system_stats?hb={int(time.time())}")
            public_failures = 0 if public_ok else public_failures + 1

        now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        pub_text = "SKIP" if public_ok is None else (f"OK/{public_status}" if public_ok else f"FAIL/{public_status}")
        print(
            f"[{now}] ComfyUI={'OK' if local_ok else 'WAIT'} | "
            f"Supervisor={'OK' if sup_ok else 'DOWN'} | Child={'OK' if child_ok else 'DOWN'} | "
            f"TunnelProc={'OK' if cf_ok else 'DOWN'} | Public={pub_text}",
            flush=True,
        )

        should_rotate = (not cf_ok) or public_failures >= PUBLIC_FAILURES_BEFORE_ROTATE
        if should_rotate and local_ok:
            print("\n♻️ Recriando Cloudflare Tunnel...", flush=True)
            ok, new_pub, new_access, status = watchdog_new_tunnel()
            if ok:
                public_failures = 0
                print("\n" + "!" * 78)
                print("✅ NOVA URL PÚBLICA")
                print(new_access)
                print("!" * 78 + "\n")
            else:
                print("⚠️ Não foi possível recriar o túnel nesta tentativa.", flush=True)

        if not sup_ok:
            print("❌ Supervisor caiu. Execute novamente a célula 8.", flush=True)
        elif not child_ok:
            print("⚠️ Supervisor está relançando o ComfyUI.", flush=True)

        time.sleep(HEARTBEAT_SECONDS)

except KeyboardInterrupt:
    print("\nKeep-alive interrompido.")
    print("Os subprocessos continuam ativos até ACTION='stop_all'.")


In [ ]:
# ============================================================
# 11) CONTROLE MANUAL — V4.4
# ============================================================

ACTION = "none"
# opções: "none", "restart_comfyui", "new_tunnel", "stop_all"

import os
import re
import signal
import subprocess
import time

if ACTION == "restart_comfyui":
    sup_pid = read_pid(SUPERVISOR_PID)
    child_pid = read_pid(COMFY_PID)
    if not pid_alive(sup_pid):
        print("Supervisor não encontrado. Execute novamente a célula 8.")
    elif not pid_alive(child_pid):
        print("ComfyUI filho não está ativo; o supervisor deve relançá-lo automaticamente.")
    else:
        print("Encerrando SOMENTE ComfyUI PID", child_pid)
        try:
            os.kill(child_pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        print("✅ Supervisor V4.4 irá relançar o ComfyUI.")

elif ACTION == "new_tunnel":
    if 'watchdog_new_tunnel' in globals():
        ok, new_pub, new_access, status = watchdog_new_tunnel()
        if ok:
            print("✅ Novo túnel:", new_access)
        else:
            print("❌ Não foi possível criar novo túnel.")
    else:
        print("A função do watchdog ainda não foi carregada. Execute a célula 10 uma vez ou execute novamente a célula 8.")

elif ACTION == "stop_all":
    STOP_FLAG.touch()
    terminate_pidfile_group(CLOUDFLARE_PID, "Cloudflare", "cloudflared", wait=2)
    terminate_pidfile_group(SUPERVISOR_PID, "Supervisor", "comfyui_supervisor.py", wait=4)
    COMFY_PID.unlink(missing_ok=True)

    # Segurança adicional contra filho órfão de versões anteriores.
    terminate_orphan_comfy()

    PUBLIC_URL_FILE.unlink(missing_ok=True)
    ACCESS_URL_FILE.unlink(missing_ok=True)
    LAUNCH_ID_FILE.unlink(missing_ok=True)
    print("✅ Supervisor, ComfyUI e Cloudflare encerrados.")

else:
    print("Nenhuma ação executada.")


## Primeiro teste da V4.4

### 1. Hugging Face gated access

Execute `1B` e `1C`.

Resultado esperado:

`Gated access : ✅ AUTORIZADO`

### 2. Patch usado pelo Manager

Depois de instalar o ComfyUI e executar `4B`, rode `7B`.

Resultado esperado:

`✅ MANAGER + HF GATED PATCH V4.4 VALIDADO`

A saída também deve conter:

`[Kaggle-HF-Gated-Patch] READY V4.4`

### 3. Launcher

Execute a célula `8`.

A V4.4 só mostra a URL pública depois de confirmar:

- Supervisor vivo;
- PID atual do ComfyUI vivo;
- `/` local respondendo;
- `/system_stats` respondendo;
- assinatura do patch HF no **log do processo real**;
- Quick Tunnel respondendo externamente.

### 4. Teste do botão de modelos ausentes

Abra o ComfyUI e faça um hard refresh uma vez.

`Modelos Ausentes → Baixar tudo`

Os downloads devem ir para `/kaggle/temp/comfyui/models/...` no servidor, e não para o computador local.

### 5. Segurança operacional

Por padrão na V4.4:

- `CORS` = `*` apenas para compatibilidade de navegação via Quick Tunnel;
- `allow_git_url_install` = false;
- `allow_pip_install` = false;
- ComfyUI escuta apenas `127.0.0.1`;
- acesso externo acontece pelo Quick Tunnel;
- instalações arbitrárias por Git URL/pip permanecem desligadas por padrão.

Se precisar instalar algo por URL Git, habilite a flag apenas durante essa operação e reinicie o ComfyUI; depois volte para `False`.

### Diagnóstico

Se qualquer validação falhar, execute a célula `9 — DIAGNÓSTICO COMPLETO` e procure primeiro por:

- `HF GATED MANAGER PATCH`;
- `Manager config`;
- `ComfyUI PID` / `Supervisor PID`;
- últimas linhas de `comfyui.log` e `comfyui-supervisor.log`.
